# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madihakomal75/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Ranked Action Queue & Trustworthy Reason Codes

We convert continuous model probabilities into plain-language, actionable editorial recommendations paired with explicit reason codes:

* **`REFRESH_URGENT`**: High predicted probability ($P \ge 0.70$) on stale, high-impression pages with falling positions.
* **`MONITOR_RANK`**: Moderate probability ($0.45 \le P < 0.70$) with recent minor position slippage.
* **`NO_ACTION_HEALTHY`**: Low probability ($P < 0.45$) indicating stable search performance and good engagement.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/madihakomal75/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier

# Load data and build features
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["staleness_ratio"] = (df["days_since_last_update"] / (df["content_age_days"] + 1)).clip(0, 1)
df["log_impressions"] = np.log1p(df["impressions_90d"].fillna(0))

features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count", "staleness_ratio", "log_impressions"]
X = df[features].fillna(df[features].median())
y = df["is_declining_label"]

# Fit model
model = HistGradientBoostingClassifier(random_state=42)
model.fit(X, y)

df["decline_probability"] = model.predict_proba(X)[:, 1]

# Assign Playbook Action & Reason Code
def assign_playbook_action(row):
    prob = row["decline_probability"]
    if prob >= 0.70:
        return "REFRESH_URGENT", "High predicted decay risk; traffic-exposed stale page."
    elif prob >= 0.45:
        return "MONITOR_RANK", "Moderate decay signal; track position weekly before rewriting."
    else:
        return "NO_ACTION_HEALTHY", "Page performing normally; preserve current content structure."

actions_and_reasons = df.apply(assign_playbook_action, axis=1)
df["action_recommendation"] = [a[0] for a in actions_and_reasons]
df["reason_code_explanation"] = [a[1] for a in actions_and_reasons]

print("Playbook Action Recommendations Distribution:")
df["action_recommendation"].value_counts()

Playbook Action Recommendations Distribution:


,count
action_recommendation,
MONITOR_RANK,11877
NO_ACTION_HEALTHY,9453
REFRESH_URGENT,8670


### Intended Use Cases & Operational Boundaries

* **Intended Use:** Decision-support tool for SEO lead editors to queue candidate URLs for content refreshes and prioritize editorial bandwidth.
* **Operational Limits:**
  * **Not a Causal Guarantee:** Updating content does not guarantee immediate rank recovery if search intent or engine algorithms shift.
  * **Seasonal Distortions:** Holiday and seasonal pages may show artificial performance drops during off-peak months; model scores should be cross-referenced with seasonal historical trends.
  * **Domain Scope:** Valid only for organic search content pages with at least 90 days of Google Search Console performance data.

In [2]:
# Limit Audit: Check performance on new pages (< 90 days age)
young_pages = df[df["content_age_days"] < 90]
print(f"Pages outside core 90-day window limits (< 90 days old): {len(young_pages):,} ({len(young_pages)/len(df):.1%})")
print("Guideline: Young pages should be excluded from automated refresh queues.")

Pages outside core 90-day window limits (< 90 days old): 0 (0.0%)
Guideline: Young pages should be excluded from automated refresh queues.


### Human-in-the-Loop Review Protocol & Mandatory No-Go Rules

Before executing any content modification based on queue recommendations, an editor must verify:

**Human Checklist:**
1. **Search Intent Verification:** Has the primary search query intent changed?
2. **Top-3 Protection (No-Go):** Never modify core structural content on pages currently ranking in **positions 1 to 3**, even if flagged stale.
3. **Conversion/Revenue Protection (No-Go):** Do not alter high-converting commercial landing pages without CRO specialist sign-off.
4. **Canonical URL Check:** Ensure page status is HTTP 200 and not redirecting.

In [3]:
# Identify No-Go candidates (Position <= 3 flagged for refresh)
no_go_candidates = df[(df["action_recommendation"] == "REFRESH_URGENT") & (df["avg_position"] <= 3.0)]

print(f"No-Go Candidates Flagged for Manual Exclusion: {len(no_go_candidates):,} pages")
if len(no_go_candidates) > 0:
    id_col = "url_hash_id" if "url_hash_id" in df.columns else df.columns[0]
    print(no_go_candidates[[id_col, "avg_position", "decline_probability", "days_since_last_update"]].head(5).to_string())

No-Go Candidates Flagged for Manual Exclusion: 393 pages
               content_id  avg_position  decline_probability  days_since_last_update
43   content_1938955b34c4           2.9             0.875314                      20
88   content_998f6f88784c           2.6             0.805492                       8
161  content_02bcf3eec147           2.2             0.888527                     104
242  content_af41d1db999a           2.9             0.924483                     104
269  content_09a3ec208780           2.9             0.795775                     104


### Model Monitoring & Retraining Triggers

To prevent recommendation decay over time, the system monitors performance and triggers retraining when:

1. **Concept Drift / Algorithm Updates:** Major core algorithm updates from Google alter rank distribution patterns.
2. **Precision Degradation:** Precision@50 on new quarterly cohorts drops below **0.65**.
3. **Distribution Shift:** The proportion of pages flagged as `REFRESH_URGENT` shifts by more than $\pm 15\%$ month-over-month.
4. **Temporal Stale Limit:** Retrain every 90 days with fresh GSC search metric cycles.

In [4]:
# Set up baseline health monitoring metrics for queue export
high_risk_ratio = (df["action_recommendation"] == "REFRESH_URGENT").mean()
print(f"Current High Risk Queue Ratio: {high_risk_ratio:.2%}")
print(f"Retrain Trigger Boundary: Alert if ratio falls below {(high_risk_ratio - 0.15):.2%} or exceeds {(high_risk_ratio + 0.15):.2%}")

Current High Risk Queue Ratio: 28.90%
Retrain Trigger Boundary: Alert if ratio falls below 13.90% or exceeds 43.90%


### Final Exports for Research Paper & Downstream Systems

We save the final ranked queue and metric summaries to `work/outputs/` for incorporation into the final report.

In [5]:
os.makedirs("work/outputs", exist_ok=True)

# Select ID column dynamically
id_col = "url_hash_id" if "url_hash_id" in df.columns else ("page_id" if "page_id" in df.columns else df.columns[0])

# Export final ranked queue
df_ranked_final = df.sort_values(by="decline_probability", ascending=False).reset_index(drop=True)
df_ranked_final["final_rank"] = df_ranked_final.index + 1

export_cols = [id_col, "final_rank", "decline_probability", "action_recommendation", "reason_code_explanation", "avg_position", "days_since_last_update", "impressions_90d"]
df_ranked_final[export_cols].to_csv("work/outputs/final_action_playbook_queue.csv", index=False)

print(f" Successfully exported final playbook queue ({len(df_ranked_final):,} rows) to work/outputs/final_action_playbook_queue.csv")
df_ranked_final[export_cols].head(5)

 Successfully exported final playbook queue (30,000 rows) to work/outputs/final_action_playbook_queue.csv


,content_id,final_rank,decline_probability,action_recommendation,reason_code_explanation,avg_position,days_since_last_update,impressions_90d
0,content_0d9c0ed65840,1,0.978211,REFRESH_URGENT,High predicted decay risk; traffic-exposed sta...,0.8,104,382
1,content_cca1e559cffd,2,0.965598,REFRESH_URGENT,High predicted decay risk; traffic-exposed sta...,2.9,20,1339
2,content_10537c63f996,3,0.963806,REFRESH_URGENT,High predicted decay risk; traffic-exposed sta...,3.5,20,370
3,content_d9e4b523c0ce,4,0.963806,REFRESH_URGENT,High predicted decay risk; traffic-exposed sta...,2.9,20,580
4,content_2d6c50388f53,5,0.963332,REFRESH_URGENT,High predicted decay risk; traffic-exposed sta...,47.4,7,14736
